In [0]:
%sql
use catalog de_workspace26;
use schema shopeasy_raw_arnav;



In [0]:
from pyspark.sql import functions as f

In [0]:
%sql
CREATE OR REPLACE VIEW gold_monthly_revenue_by_region AS
SELECT
  YEAR(order_date) AS year,
  MONTH(order_date) AS month,
  region,
  SUM(revenue) AS total_revenue
FROM silver_orders
GROUP BY 1, 2, 3;


select * from gold_monthly_revenue_by_region;

In [0]:
%sql
CREATE OR REPLACE VIEW gold_top_products AS
SELECT
  product_name, category,
  SUM(revenue) AS total_revenue,
  RANK() OVER (PARTITION BY category ORDER BY SUM(revenue) DESC) AS rank
FROM silver_orders
GROUP BY product_name, category;

In [0]:

streaming_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option('cloudFiles.schemaLocation', "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_schemas/gold_live_orders")\
  .option("header", "true")
  .option("inferSchema", "true")
  .load("/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/"))

from pyspark.sql.functions import to_timestamp

streaming_df = streaming_df \
  .withColumn("order_date", to_timestamp(f.col("order_date"), "yyyy-MM-dd")) \
  .withWatermark("order_date", "10 minutes")

In [0]:
query = (streaming_df.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_checkpoints/gold_live")
  .trigger(once=True)\
  .toTable("gold_live_orders"))

query.awaitTermination()

In [0]:
count_before = spark.read.table("gold.live_orders").count()
query.stop()

count_after = spark.read.table("gold.live_orders").count()
print(f"Before: {count_before}, After: {count_after}") 

if count_before == count_after:
    print(f"   PASSED — No duplicate rows written.")
    print(f"     Checkpoint correctly tracked all processed files.")
    print(f"     On restart, Spark skipped already processed files.")
else:
    diff = count_after - count_before
    print(f"  FAILED — {diff} extra rows found!")
    print(f"     Ensure checkpointLocation is identical on both runs.")

In [0]:
%sql
VACUUM silver.orders RETAIN 168 HOURS;
